# DID Study on the Effects of ECPill

## Libraries & Configurations

In [1]:
# Clear the workspace
rm(list = ls())

In [2]:
# Directories for data and output
DIR_DATA   <- "Data/"
DIR_OUT <- "Output/"
dir.create(DIR_OUT, showWarnings = FALSE, recursive = TRUE)

In [3]:
# Required packages
# install.packages("tidyverse")
# install.packages("purrr")
# install.packages("tibble")
# install.packages("fixest")
# install.packages("broom")
# install.packages("TwoWayFEWeights")
# install.packages("remotes")
# install.packages("did")
# remotes::install_github("s3alfisc/fwildclusterboot")

library(tidyverse)
library(glue)
library(purrr)
library(tibble)
library(fixest)
library(fwildclusterboot)
library(broom)
library(did)
library(TwoWayFEWeights)

# Silence fixest's collinearity/removed-fixed-effects notes
fixest::setFixest_notes(FALSE)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
“package ‘fwildclusterboot’ was built under R version 4.5.3”

Please cite as: 


 Fischer & Roodman. (2021). fwildclusterboot: Fast Wild Cluster.

 Bootstrap Inference for Linear Regression Models.

 Available from https://cran.r-project.org/package=fwildclusterboot/.



## Data Preprocessing

### (1) Load raw data

In [4]:
data_raw_births <- read.csv(paste0(DIR_DATA, "S1Data_granular_covars.csv"), stringsAsFactors = FALSE)
data_raw_deaths <- read.csv(paste0(DIR_DATA, "S1Data_deaths_covars.csv"), stringsAsFactors = FALSE)

cat(glue("
Summary of raw data:
- Births data summary:
  - Number of rows: {nrow(data_raw_births)}
  - Number of distinct municipalities: {n_distinct(data_raw_births$dom_comuna)}
- Deaths data summary:
  - Number of rows: {nrow(data_raw_deaths)}
  - Number of distinct municipalities: {n_distinct(data_raw_deaths$dom_comuna)}
\n"))

Summary of raw data:
- Births data summary:
  - Number of rows: 209056
  - Number of distinct municipalities: 336
- Deaths data summary:
  - Number of rows: 209056
  - Number of distinct municipalities: 336


### (2) Data processing: Aggregate granular data into a municipality x year panel

Data structure after aggregation:
- Identifiers: dom_comuna, year
- Key Variables:

    - `popln`: Population within the age range
    - `births`: Number of births within the age range
    - `pill`: Indicator for pill usage
    - `pilldistance`: Distance to the nearest pill distribution point
    - `early_deaths`: Number of early deaths within the age range
    - `birth_rate`: Birth rate per 1000 women
    - `death_rate`: Early death rate per 1000 live births
    - `trend`: Linear time trend within each municipality
    - `close10`: Indicator for being close to the nearest treated municipality within 10 km (requires pill == 0)
    - `close20`: Indicator for being close to the nearest treated municipality within 10-20 km (requires pill == 0)
    - `close30`: Indicator for being close to the nearest treated municipality within 20-30 km (requires pill == 0)

- Covariates:

    - `X_party_` Series: Dummy variables for the mayor's political party
    - `X_mujer`: Indicator for whether the mayor is female
    - `X_votop`: The mayor's vote share/margin
    - `X_outofschool`: Percentage of girls out of high school
    - `X_educationspend`: Total education spending
    - `X_educationmunicip`: Municipal-level education spending
    - `X_healthspend`: Total health spending
    - `X_healthtraining`: Health worker training spending
    - `X_healthstaff`: Health staffing spending
    - `X_femalepoverty`: Percentage of female-headed households below the poverty line
    - `X_femaleworkers`: Percentage of female workers in professional/public positions
    - `X_condom`: Condom availability/usage rate (regional level)
    - `X_region_` Series: Dummy variables for the municipality's region

In [5]:
# Covariates:
X_cols <- c(
  "party", "mujer", "votop",                           # Mayor-related variables
  "outofschool", "educationspend", "educationmunicip", # Education-related variables
  "healthspend", "healthtraining", "healthstaff",      # Health-related variables
  "femalepoverty", "femaleworkers",                    # Female-related variables
  "condom",                                            # Contraceptive use
  "region"                                             # Region code
)

In [6]:
# Function to prepare model data for a specific age range
get_model_data <- function(age_lo, age_hi) {

  # Temporary function to create dummy variables for categorical columns: 'party' and 'region'
  make_dummies <- function(df, col) {
    lvls <- sort(unique(df[[col]]))
    for (lv in lvls) {
      df[[paste0("X_", col, "_", lv)]] <- as.integer(df[[col]] == lv)
    }
    df[[col]] <- NULL
    df
  }

  # Aggregate births data by municipality and year
  agg_births <- data_raw_births |>
    filter(age >= age_lo, age <= age_hi) |>
    group_by(dom_comuna, year) |>
    summarise(
      popln        = sum(n, na.rm = TRUE),
      births       = sum(n[pregnant == 1], na.rm = TRUE),
      pill         = first(pill),
      pilldistance = first(pilldistance),
      across(all_of(X_cols), ~first(.x)),
      .groups = "drop"
    ) |>
    make_dummies("party") |>
    make_dummies("region") |>
    rename_with(~ paste0("X_", .x), .cols = setdiff(X_cols, c("party", "region")))

  # Aggregate deaths data by municipality and year
  agg_deaths <- data_raw_deaths |>
    filter(age >= age_lo, age <= age_hi) |>
    group_by(dom_comuna, year) |>
    summarise(
      early_deaths = sum(earlyP, na.rm = TRUE),
      .groups = "drop"
    )

  # Aggregate combined data by municipality and year
  agg_combine <- agg_births |> inner_join(agg_deaths, by = c("dom_comuna", "year")) |>
    arrange(dom_comuna, year) |>
    group_by(dom_comuna) |>
    mutate(trend = year - min(year)) |>
    ungroup() |>
    mutate(
      birth_rate = (births / popln) * 1000,
      death_rate = (early_deaths / births) * 1000,
      dom_comuna = factor(dom_comuna),
      close10 = as.integer(pill == 0 & pilldistance >  0 & pilldistance <= 10),
      close20 = as.integer(pill == 0 & pilldistance > 10 & pilldistance <= 20),
      close30 = as.integer(pill == 0 & pilldistance > 20 & pilldistance <= 30)
    ) |>
    relocate(starts_with("X_"), .after = last_col())

  # Return the aggregated combined data
  return(agg_combine)
}

In [7]:
# Create model data for 3 different age groups
data_age1 <- get_model_data(15, 49)   # All women
data_age2 <- get_model_data(15, 19)   # Teens 15-19 years old
data_age3 <- get_model_data(20, 34)   # Young adults 20-34 years old

# Summary of aggregated data
cat(glue("
Summary of aggregated data:
- Group 1: All Women
  - Number of municipalities: {n_distinct(data_age1$dom_comuna)}
  - Year range: {min(data_age1$year)}-{max(data_age1$year)}
  - Number of years: {length(unique(data_age1$year))}
  - Number of observations: {nrow(data_age1)}
- Group 2: Teens
  - Number of municipalities: {n_distinct(data_age2$dom_comuna)}
  - Year range: {min(data_age2$year)}-{max(data_age2$year)}
  - Number of years: {length(unique(data_age2$year))}
  - Number of observations: {nrow(data_age2)}
- Group 3: Young Adults
  - Number of municipalities: {n_distinct(data_age3$dom_comuna)}
  - Year range: {min(data_age3$year)}-{max(data_age3$year)}
  - Number of years: {length(unique(data_age3$year))}
  - Number of observations: {nrow(data_age3)}
\n"))

Summary of aggregated data:
- Group 1: All Women
  - Number of municipalities: 336
  - Year range: 2006-2011
  - Number of years: 6
  - Number of observations: 1934
- Group 2: Teens
  - Number of municipalities: 335
  - Year range: 2006-2011
  - Number of years: 6
  - Number of observations: 1929
- Group 3: Young Adults
  - Number of municipalities: 336
  - Year range: 2006-2011
  - Number of years: 6
  - Number of observations: 1934


## Stage 1: TWFE Replication + De Chaisemartin & D'Haultfoeuille (2020) Diagnostic

### (1) Stage 1 Task 1: TWFE Replication

In [8]:
# Function of calculating standard errors: Use wild cluster bootstrap and fallback to cluster-robust (CRV1)
run_boottest <- function(model, param, cluster_var, spec_name) {
  
  # Try to run the wild cluster bootstrap first
  boot_obj <- tryCatch(
    {
      # capture.output() swallows boottest()'s own printed diagnostics (e.g. its
      # "confidence inversion failed" notice) -- these are plain printed text,
      # not warnings/messages, so suppressWarnings()/suppressMessages() alone
      # can't catch them
      invisible(capture.output(
        boot_result <- suppressWarnings(suppressMessages(
          boottest(model, param = param, clustid = cluster_var, B = 999)
        ))
      ))
      boot_result
    },
    error = function(e) {
      cat(glue("[{spec_name}] failed to run wild cluster bootstrap({conditionMessage(e)}), and change to cluster-robust(CRV1) standard errors\n"))
      return(NULL)
    }
  )

  # If the wild cluster bootstrap does not fail, extract the results and format them into a tibble
  if (!is.null(boot_obj)) {
    bt <- broom::tidy(boot_obj)  

    result <- tibble(
      coef      = bt$estimate[1],
      se        = bt$estimate[1] / bt$statistic[1],
      p_value   = bt$p.value[1],
      ci_lower  = bt$conf.low[1],
      ci_upper  = bt$conf.high[1],
      se_method = "wild_cluster_bootstrap"
    )
  }
  # If the wild cluster bootstrap fails, fall back to cluster-robust standard errors (CRV1)
  else {
    s  <- summary(model, cluster = cluster_var)
    ct <- s$coeftable[param, ]
    ci <- confint(s)[param, ]
    result <- tibble(
      coef      = unname(ct[["Estimate"]]),
      se        = unname(ct[["Std. Error"]]),
      p_value   = unname(ct[["Pr(>|t|)"]]),
      ci_lower  = unname(ci[[1]]),
      ci_upper  = unname(ci[[2]]),
      se_method = "cluster_robust_CRV1_fallback"
    )
  }

  return(result)
}

In [9]:
# Function to run TWFE regression with optional population weighting and wild cluster bootstrap for standard errors
run_twfe <- function(df, outcome, model_spec) {

  # Data for the model
  df <- df |>
    filter(is.finite(.data[[outcome]])) |>
    group_by(dom_comuna) |>
    filter(n() > 1) |>
    ungroup() |>
    mutate(dom_comuna = droplevels(dom_comuna))

  # Define the base and trend formulas for the TWFE regression models
  f_base  <- as.formula(paste(outcome, "~ pill | dom_comuna + year"))
  f_trend <- as.formula(paste(outcome, "~ pill + i(dom_comuna, trend) | dom_comuna + year"))

  # Three TWFE regression models: baseline, with trends, and population-weighted baseline
  m_base     <- feols(f_base,  data = df)
  m_trend    <- feols(f_trend, data = df)
  m_base_pw  <- feols(f_base,  data = df, weights = ~popln)

  set.seed(2727)  # Fixed seed for reproducibility of bootstrap results

  # Function to extract results from a model using wild cluster bootstrap
  extract <- function(model, spec_name) {
    run_boottest(model, "pill", "dom_comuna", spec_name) |>
      mutate(
        model_spec  = model_spec,
        outcome     = outcome,
        spec        = spec_name,
        n_obs       = nobs(model),
        n_municipal = n_distinct(df$dom_comuna),
        .before = 1
      )
  }

  # Run the extraction function on each of the three TWFE models
  result <- bind_rows(
    extract(m_base,    "baseline_TWFE"),
    extract(m_trend,   "TWFE_with_trends"),
    extract(m_base_pw, "baseline_popln_weighted")
  )

  # Return the combined results from all three models
  return(result)
}

In [10]:
# Run the TWFE regressions for all 3 age groups and 2 outcomes (6 groups)
twfe_11 <- run_twfe(data_age1, "birth_rate", "Birth Rate & All Age")     |> mutate(age_group = "all_women",   .before = 1)
twfe_12 <- run_twfe(data_age1, "death_rate", "Death Rate & All Age")     |> mutate(age_group = "all_women",   .before = 1)
twfe_21 <- run_twfe(data_age2, "birth_rate", "Birth Rate & Teens")       |> mutate(age_group = "teens",       .before = 1)
twfe_22 <- run_twfe(data_age2, "death_rate", "Death Rate & Teens")       |> mutate(age_group = "teens",       .before = 1)
twfe_31 <- run_twfe(data_age3, "birth_rate", "Birth Rate & Young Adult") |> mutate(age_group = "young_adult", .before = 1)
twfe_32 <- run_twfe(data_age3, "death_rate", "Death Rate & Young Adult") |> mutate(age_group = "young_adult", .before = 1)

results_s1t1_twfe_origin <- bind_rows(twfe_11, twfe_12, twfe_21, twfe_22, twfe_31, twfe_32)
write.csv(results_s1t1_twfe_origin, paste0(DIR_OUT, "results_s1t1_twfe_origin.csv"), row.names = FALSE)

cat("\n===== Stage 1 Task 1: Baseline TWFE Estimates (6 Groups) =====\n")
results_s1t1_twfe_origin |> filter(spec == "baseline_TWFE")


===== Stage 1 Task 1: Baseline TWFE Estimates (6 Groups) =====


age_group,model_spec,outcome,spec,n_obs,n_municipal,coef,se,p_value,ci_lower,ci_upper,se_method
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
all_women,Birth Rate & All Age,birth_rate,baseline_TWFE,1934,336,-0.91221563,0.6436792,0.18618619,-2.1889607,0.37312745,wild_cluster_bootstrap
all_women,Death Rate & All Age,death_rate,baseline_TWFE,1913,333,0.01824187,0.2905744,0.95295295,-0.5646155,0.60091195,wild_cluster_bootstrap
teens,Birth Rate & Teens,birth_rate,baseline_TWFE,1929,335,-1.46916742,1.1297139,0.19519520,-3.6279238,0.68568970,wild_cluster_bootstrap
teens,Death Rate & Teens,death_rate,baseline_TWFE,1887,330,-1.33268037,0.6455774,0.03603604,-2.5935211,-0.07420367,wild_cluster_bootstrap
young_adult,Birth Rate & Young Adult,birth_rate,baseline_TWFE,1934,336,-1.61081556,1.1563033,0.15515516,-3.8072612,0.58993034,wild_cluster_bootstrap
young_adult,Death Rate & Young Adult,death_rate,baseline_TWFE,1910,333,0.52619926,0.3330711,0.13213213,-0.1511625,1.19961460,wild_cluster_bootstrap


### (2) Stage 1 Task 2: De Chaisemartin & D'Haultfoeuille (2020) Diagnostics

In [11]:
# Diagnostic function
run_diagnostic <- function(df, outcome, model_spec) {
  
  # Data for the diagnostic function
  df <- df |>
    filter(is.finite(.data[[outcome]])) |>
    group_by(dom_comuna) |>
    filter(n() > 1) |>
    ungroup()

  # Diagnostic using TwoWayFEWeights
  diag <- TwoWayFEWeights::twowayfeweights(
    data              = as.data.frame(df),
    Y                 = outcome,
    G                 = "dom_comuna",
    T                 = "year",
    D                 = "pill",
    type              = "feTR",     # Use the "feTR" type for the TwoWayFEWeights diagnostic
    summary_measures  = TRUE        # Return summary measures for the diagnostic
  )

  # Summarize the diagnostic results into a tibble
  result <- tibble(
    model_spec     = model_spec,
    outcome        = outcome,
    sum_plus       = diag$sum_plus,                                  # Sum of positive weights
    sum_minus      = diag$sum_minus,                                 # Sum of negative weights
    share_negative = diag$nr_minus / (diag$nr_plus + diag$nr_minus), # Share of negative weights (by number of ATT(g,t) cells)
    min_sd_flip    = diag$sensibility                                # Minimum treatment effect standard deviation (Corollary 1)
  )

  return(result)
}

In [12]:
# Run the diagnostic function for all 3 age groups and 2 outcomes (6 groups)
diag_11 <- run_diagnostic(data_age1, "birth_rate", "Birth Rate & All Age")     |> mutate(age_group = "all_women",   .before = 1)
diag_12 <- run_diagnostic(data_age1, "death_rate", "Death Rate & All Age")     |> mutate(age_group = "all_women",   .before = 1)
diag_21 <- run_diagnostic(data_age2, "birth_rate", "Birth Rate & Teens")       |> mutate(age_group = "teens",       .before = 1)
diag_22 <- run_diagnostic(data_age2, "death_rate", "Death Rate & Teens")       |> mutate(age_group = "teens",       .before = 1)
diag_31 <- run_diagnostic(data_age3, "birth_rate", "Birth Rate & Young Adult") |> mutate(age_group = "young_adult", .before = 1)
diag_32 <- run_diagnostic(data_age3, "death_rate", "Death Rate & Young Adult") |> mutate(age_group = "young_adult", .before = 1)

results_s1t2_diagnostic <- bind_rows(diag_11, diag_12, diag_21, diag_22, diag_31, diag_32)
write.csv(results_s1t2_diagnostic,   paste0(DIR_OUT, "results_s1t2_diagnostic.csv"), row.names = FALSE)

cat("\n===== Stage 1 Task 2: De Chaisemartin & D'Haultfoeuille Diagnostics =====\n")
results_s1t2_diagnostic


===== Stage 1 Task 2: De Chaisemartin & D'Haultfoeuille Diagnostics =====


age_group,model_spec,outcome,sum_plus,sum_minus,share_negative,min_sd_flip
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
all_women,Birth Rate & All Age,birth_rate,1,0,0,3.18298164
all_women,Death Rate & All Age,death_rate,1,0,0,0.06459844
teens,Birth Rate & Teens,birth_rate,1,0,0,5.13896313
teens,Death Rate & Teens,death_rate,1,0,0,4.62338926
young_adult,Birth Rate & Young Adult,birth_rate,1,0,0,5.62059693
young_adult,Death Rate & Young Adult,death_rate,1,0,0,1.84638581


### (3) Stage 1 Task 3: Spillover Robustness Check

In [13]:
# Spillover robustness check function
run_spillover <- function(df, outcome, model_spec) {
  
  # Data for the spillover robustness check
  df <- df |>
    filter(is.finite(.data[[outcome]])) |>
    group_by(dom_comuna) |>
    filter(n() > 1) |>
    ungroup() |>
    mutate(dom_comuna = droplevels(dom_comuna))

  # Formula and model for the spillover robustness check
  f_sp <- as.formula(paste(outcome, "~ pill + close10 + close20 + close30 | dom_comuna + year"))
  m_sp <- feols(f_sp, data = df)

  # Set seed for reproducibility of the bootstrap test
  set.seed(2727)

  # Run the bootstrap test for the pill coefficient with spillover close bins
  pill_row <- run_boottest(m_sp, "pill", "dom_comuna", "spillover_close_bins") |>
    mutate(
      model_spec = model_spec,
      outcome = outcome,
      spec = "spillover_close_bins",
      n_obs = nobs(m_sp),
      n_municipal = n_distinct(df$dom_comuna),
      .before = 1
    )

  # Extract the coefficients for the close10/20/30 variables with cluster-robust SEs
  close_coefs <- summary(m_sp, cluster = "dom_comuna")$coeftable
  close_coefs <- close_coefs[grepl("^close", rownames(close_coefs)), , drop = FALSE]

  # Return a list containing the pill coefficient row and the close10/20/30 coefficients
  result <- list(pill = pill_row, close = close_coefs)

  return(result)
}

In [14]:
# Run the spillover robustness checks for all age groups and outcomes
sp_11 <- run_spillover(data_age1, "birth_rate", "Birth Rate & All Age")
sp_12 <- run_spillover(data_age1, "death_rate", "Death Rate & All Age")
sp_21 <- run_spillover(data_age2, "birth_rate", "Birth Rate & Teens")
sp_22 <- run_spillover(data_age2, "death_rate", "Death Rate & Teens")
sp_31 <- run_spillover(data_age3, "birth_rate", "Birth Rate & Young Adult")
sp_32 <- run_spillover(data_age3, "death_rate", "Death Rate & Young Adult")

results_s1t3_spillover <- bind_rows(
  sp_11$pill |> mutate(age_group = "all_women",   .before = 1),
  sp_12$pill |> mutate(age_group = "all_women",   .before = 1),
  sp_21$pill |> mutate(age_group = "teens",       .before = 1),
  sp_22$pill |> mutate(age_group = "teens",       .before = 1),
  sp_31$pill |> mutate(age_group = "young_adult", .before = 1),
  sp_32$pill |> mutate(age_group = "young_adult", .before = 1)
)

write.csv(results_s1t3_spillover, paste0(DIR_OUT, "results_s1t3_spillover.csv"), row.names = FALSE)

cat("\n===== Stage 1 Task 3: Spillover Robustness Checks =====\n")
results_s1t3_spillover


===== Stage 1 Task 3: Spillover Robustness Checks =====


age_group,model_spec,outcome,spec,n_obs,n_municipal,coef,se,p_value,ci_lower,ci_upper,se_method
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
all_women,Birth Rate & All Age,birth_rate,spillover_close_bins,1934,336,0.007532637,1.1382218,0.9959960,NA,NA,wild_cluster_bootstrap
all_women,Death Rate & All Age,death_rate,spillover_close_bins,1913,333,0.149090065,0.3389862,0.7027027,-0.5364381,0.8296102,wild_cluster_bootstrap
teens,Birth Rate & Teens,birth_rate,spillover_close_bins,1929,335,-0.957076289,1.9042893,0.6276276,-4.8521019,2.9907898,wild_cluster_bootstrap
teens,Death Rate & Teens,death_rate,spillover_close_bins,1887,330,-1.733882201,1.2082842,0.1691692,-4.0578250,0.5659340,wild_cluster_bootstrap
young_adult,Birth Rate & Young Adult,birth_rate,spillover_close_bins,1934,336,-0.934962608,1.9443267,0.6416416,-4.7582455,3.1592735,wild_cluster_bootstrap
young_adult,Death Rate & Young Adult,death_rate,spillover_close_bins,1910,333,0.469844898,0.3393275,0.1801802,-0.2076620,1.1601102,wild_cluster_bootstrap


## Stage 2: TWFE on Non-Switcher Municipalities

In [15]:
# Function to identify and remove "switcher" municipalities: Callaway & Sant'Anna (2021)
remove_switchers <- function(panel, age_group) {
  # Identify switcher municipalities
  switcher_info <- panel |>
    arrange(dom_comuna, year) |>
    group_by(dom_comuna) |>
    summarise(is_switcher = as.integer(any(diff(pill) == -1)), .groups = "drop")

  # Count the number of switcher and non-switcher municipalities
  non_switcher_ids <- switcher_info |>
    filter(is_switcher == 0) |> 
    pull(dom_comuna)

  n_switch <- sum(switcher_info$is_switcher)
  n_total  <- nrow(switcher_info)
  cat(glue("
  [{age_group}] Identified {n_switch}/{n_total} switcher municipalities; {n_total - n_switch} remain in the restricted sample.
  \n"))

  # Remove switcher municipalities from the panel
  result_panel <- panel |> 
    filter(dom_comuna %in% non_switcher_ids)

  return(result_panel)
}

In [16]:
# Apply to all 3 age groups to get switcher-excluded restricted data
data_age1_ns <- remove_switchers(data_age1, "All Women")
data_age2_ns <- remove_switchers(data_age2, "Teens")
data_age3_ns <- remove_switchers(data_age3, "Young Adults")

[All Women] Identified 36/336 switcher municipalities; 300 remain in the restricted sample.
[Teens] Identified 36/335 switcher municipalities; 299 remain in the restricted sample.
[Young Adults] Identified 36/336 switcher municipalities; 300 remain in the restricted sample.


In [17]:
# Run the same TWFE regressions on the switcher-excluded restricted data
twfe_ns_11 <- run_twfe(data_age1_ns, "birth_rate", "Birth Rate & All Age")     |> mutate(age_group = "all_women",   .before = 1)
twfe_ns_12 <- run_twfe(data_age1_ns, "death_rate", "Death Rate & All Age")     |> mutate(age_group = "all_women",   .before = 1)
twfe_ns_21 <- run_twfe(data_age2_ns, "birth_rate", "Birth Rate & Teens")       |> mutate(age_group = "teens",       .before = 1)
twfe_ns_22 <- run_twfe(data_age2_ns, "death_rate", "Death Rate & Teens")       |> mutate(age_group = "teens",       .before = 1)
twfe_ns_31 <- run_twfe(data_age3_ns, "birth_rate", "Birth Rate & Young Adult") |> mutate(age_group = "young_adult", .before = 1)
twfe_ns_32 <- run_twfe(data_age3_ns, "death_rate", "Death Rate & Young Adult") |> mutate(age_group = "young_adult", .before = 1)

results_s2t1_twfe_ns <- bind_rows(twfe_ns_11, twfe_ns_12, twfe_ns_21, twfe_ns_22, twfe_ns_31, twfe_ns_32)

write.csv(results_s2t1_twfe_ns, paste0(DIR_OUT, "results_s2t1_twfe_ns.csv"), row.names = FALSE)

cat("\n===== Stage 2 Task 1: TWFE Estimates on Switcher-Excluded Restricted Sample (6 Groups) =====\n")
results_s2t1_twfe_ns |> filter(spec == "baseline_TWFE")


===== Stage 2 Task 1: TWFE Estimates on Switcher-Excluded Restricted Sample (6 Groups) =====


age_group,model_spec,outcome,spec,n_obs,n_municipal,coef,se,p_value,ci_lower,ci_upper,se_method
<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
all_women,Birth Rate & All Age,birth_rate,baseline_TWFE,1718,300,-1.0043079,0.7965324,0.2032032,-2.4788632,0.4748933,wild_cluster_bootstrap
all_women,Death Rate & All Age,death_rate,baseline_TWFE,1697,297,-0.1837025,0.3201229,0.5795796,-0.7905247,0.4439166,wild_cluster_bootstrap
teens,Birth Rate & Teens,birth_rate,baseline_TWFE,1713,299,-1.5645226,1.3529381,0.2622623,-4.1807750,1.0420217,wild_cluster_bootstrap
teens,Death Rate & Teens,death_rate,baseline_TWFE,1671,294,-1.1374391,0.8198722,0.1791792,-2.7523744,0.4416059,wild_cluster_bootstrap
young_adult,Birth Rate & Young Adult,birth_rate,baseline_TWFE,1718,300,-2.0082750,1.4550923,0.1871872,-4.9660699,0.8941263,wild_cluster_bootstrap
young_adult,Death Rate & Young Adult,death_rate,baseline_TWFE,1694,297,0.4670973,0.3791315,0.2032032,-0.2795719,1.2032023,wild_cluster_bootstrap


## Stage 3: DID using Callaway & Sant'Anna (2021) methods

### (1) Stage 3 Task 1: Callaway & Sant'Anna (2021) Estimation

In [18]:
# Add first-treatment year (gname required by the did package)
add_first_treat <- function(panel) {
  # Compute the first treatment year for each unit (dom_comuna)
  first_treat <- panel |>
    group_by(dom_comuna) |>
    summarise(first_treat = if (any(pill == 1)) min(year[pill == 1]) else 0, .groups = "drop")
  
  # Merge the first treatment year back into the original panel and create a numeric id for the did package
  result <- panel |>
    left_join(first_treat, by = "dom_comuna") |>
    mutate(id_num = as.integer(factor(dom_comuna)))   # did package expects a numeric id
  
  return(result)
}

In [19]:
# Function to run the Callaway & Sant'Anna (2021)
run_cs <- function(panel, outcome, age_group, control_group = "nevertreated", weights_col = NULL) {

  # Add the first treatment indicator to the panel data
  dat <- add_first_treat(panel)

  # Run the Callaway & Sant'Anna (2021) estimator on the prepared panel data
  invisible(capture.output(
    att <- suppressWarnings(suppressMessages(att_gt(
      yname                  = outcome,
      tname                  = "year",
      idname                 = "id_num",
      gname                  = "first_treat",
      xformla                = ~1,               # No covariates: TWFE with regressors can be severely biased
      data                   = as.data.frame(dat),
      control_group          = control_group,
      weightsname            = weights_col,       # NULL = unweighted (main spec), "popln" = robustness check
      allow_unbalanced_panel = TRUE,
      clustervars            = "id_num",
      bstrap                 = TRUE,
      biters                 = 1000,
      base_period            = "varying"
    )))
  ))

  # Aggregate the ATT estimates using the "simple" method
  simple_agg <- aggte(att, type = "simple")

  # Compute the p-value for the overall ATT estimate using a normal approximation
  p_value <- 2 * pnorm(-abs(simple_agg$overall.att / simple_agg$overall.se))

  # Store the results in a list for easy access and further analysis
  result <- list(
    att_gt     = att,
    simple     = simple_agg,
    dynamic    = aggte(att, type = "dynamic"),
    group      = aggte(att, type = "group"),
    simple_row = tibble(
      age_group = age_group, 
      outcome = outcome,
      coef = simple_agg$overall.att, 
      se = simple_agg$overall.se,
      p_value = p_value
    )
  )
  
  return(result)
}

In [20]:
# Run the Callaway & Sant'Anna estimation for all 3 age groups and 2 outcomes (6 groups)
cs_11 <- run_cs(data_age1_ns, "birth_rate", "all_women")
cs_12 <- run_cs(data_age1_ns, "death_rate", "all_women")
cs_21 <- run_cs(data_age2_ns, "birth_rate", "teens")
cs_22 <- run_cs(data_age2_ns, "death_rate", "teens")
cs_31 <- run_cs(data_age3_ns, "birth_rate", "young_adult")
cs_32 <- run_cs(data_age3_ns, "death_rate", "young_adult")

results_s3t1_cs <- bind_rows(
  cs_11$simple_row, cs_12$simple_row,
  cs_21$simple_row, cs_22$simple_row,
  cs_31$simple_row, cs_32$simple_row
)

write.csv(results_s3t1_cs, paste0(DIR_OUT, "results_s3t1_cs.csv"), row.names = FALSE)

cat("\n===== Stage 3 Task 1: Callaway & Sant'Anna (2021) Estimates (6 Groups) =====\n")
results_s3t1_cs


===== Stage 3 Task 1: Callaway & Sant'Anna (2021) Estimates (6 Groups) =====


age_group,outcome,coef,se,p_value
<chr>,<chr>,<dbl>,<dbl>,<dbl>
all_women,birth_rate,-0.2533298,1.1942396,0.8320084
all_women,death_rate,-0.2139017,0.4231297,0.6131917
teens,birth_rate,0.1718972,2.6586104,0.9484473
teens,death_rate,-0.5878682,1.1987108,0.6238388
young_adult,birth_rate,-0.8486241,1.9023432,0.6555293
young_adult,death_rate,0.3985831,0.4729204,0.3993335


### (2) Stage 3 Task 2: Pre-treatment Dynamic Effects (Placebo-style Pre-trends Check)

In [ ]:
# Extract pre-treatment (e<0) dynamic effects as a "placebo-style" pre-trends check
extract_pretrends <- function(cs_obj, age_group, outcome) {
  # Extract the dynamic ATT estimates from the Callaway & Sant'Anna (2021) object
  d <- cs_obj$dynamic
  
  # Create a tibble with the relevant dynamic ATT estimates and their standard errors
  result <- 
    tibble(
      age_group  = age_group,
      outcome    = outcome,
      event_time = d$egt,
      att        = d$att.egt,
      se         = d$se.egt,
      crit_val   = d$crit.val.egt
    ) |>
    filter(event_time < 0) |>
    mutate(sig_at_5pct = abs(att / se) > crit_val)
  
  return(result)
}

In [22]:
# Apply the function to all relevant Callaway & Sant'Anna (2021) objects and combine the results
results_s3t2_pretrends <- bind_rows(
  extract_pretrends(cs_11, "all_women",   "birth_rate"),
  extract_pretrends(cs_12, "all_women",   "death_rate"),
  extract_pretrends(cs_21, "teens",       "birth_rate"),
  extract_pretrends(cs_22, "teens",       "death_rate"),
  extract_pretrends(cs_31, "young_adult", "birth_rate"),
  extract_pretrends(cs_32, "young_adult", "death_rate")
)

write.csv(results_s3t2_pretrends, paste0(DIR_OUT, "results_s3t2_pretrends.csv"), row.names = FALSE)

cat("\n===== Stage 3 Task 2: Pre-treatment Dynamic Effects (Placebo-style Pre-trends Check) =====\n")
results_s3t2_pretrends


===== Stage 3 Task 2: Pre-treatment Dynamic Effects (Placebo-style Pre-trends Check) =====


age_group,outcome,event_time,att,se,crit_val,sig_at_5pct
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
all_women,birth_rate,-4,1.0844315664,1.0058154,2.566950,FALSE
all_women,birth_rate,-3,0.7131711737,0.6515717,2.566950,FALSE
all_women,birth_rate,-2,-2.1478701053,0.8790663,2.566950,FALSE
all_women,birth_rate,-1,1.2501697375,0.9600915,2.566950,FALSE
all_women,death_rate,-4,-0.1222579264,0.5120238,2.540974,FALSE
all_women,death_rate,-3,0.1340838763,0.3690440,2.540974,FALSE
all_women,death_rate,-2,0.1735503506,0.4208195,2.540974,FALSE
all_women,death_rate,-1,-0.8157093504,0.4248833,2.540974,FALSE
teens,birth_rate,-4,-0.1144422503,2.2771009,2.474979,FALSE


## Comparison of Results of 3 Stages

In [23]:
# Build overall comparison table of 3 stages
result_overall_comparison <- results_s1t1_twfe_origin |>
  filter(spec == "baseline_TWFE") |>
  select(age_group, outcome, TWFE_full_coef = coef, TWFE_full_se = se, TWFE_full_pvalue = p_value) |>
  left_join(
    results_s2t1_twfe_ns |>
      filter(spec == "baseline_TWFE") |>
      select(age_group, outcome, TWFE_restricted_coef = coef, TWFE_restricted_se = se, TWFE_restricted_pvalue = p_value),
    by = c("age_group", "outcome")
  ) |>
  left_join(
    results_s3t1_cs |>
      select(age_group, outcome, CS_restricted_coef = coef, CS_restricted_se = se, CS_restricted_pvalue = p_value),
    by = c("age_group", "outcome")
  )

write.csv(result_overall_comparison, paste0(DIR_OUT, "result_overall_comparison.csv"), row.names = FALSE)

cat("\n===== Comparison of Results of 3 Stages =====\n")
result_overall_comparison


===== Comparison of Results of 3 Stages =====


age_group,outcome,TWFE_full_coef,TWFE_full_se,TWFE_full_pvalue,TWFE_restricted_coef,TWFE_restricted_se,TWFE_restricted_pvalue,CS_restricted_coef,CS_restricted_se,CS_restricted_pvalue
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
all_women,birth_rate,-0.91221563,0.6436792,0.18618619,-1.0043079,0.7965324,0.2032032,-0.2533298,1.1942396,0.8320084
all_women,death_rate,0.01824187,0.2905744,0.95295295,-0.1837025,0.3201229,0.5795796,-0.2139017,0.4231297,0.6131917
teens,birth_rate,-1.46916742,1.1297139,0.19519520,-1.5645226,1.3529381,0.2622623,0.1718972,2.6586104,0.9484473
teens,death_rate,-1.33268037,0.6455774,0.03603604,-1.1374391,0.8198722,0.1791792,-0.5878682,1.1987108,0.6238388
young_adult,birth_rate,-1.61081556,1.1563033,0.15515516,-2.0082750,1.4550923,0.1871872,-0.8486241,1.9023432,0.6555293
young_adult,death_rate,0.52619926,0.3330711,0.13213213,0.4670973,0.3791315,0.2032032,0.3985831,0.4729204,0.3993335
